In [ ]:
"""Single-notebook source for BigAlpha 2026 AI factor private leaderboard.

The platform extracts this code and calls main(datasources, start_date, end_date).
No local cache, model weight, external file or network service is required.
"""


def main(datasources, start_date, end_date):
    import gc
    import math
    import os
    import random

    os.environ.setdefault("POLARS_MAX_THREADS", "16")
    os.environ.setdefault("OMP_NUM_THREADS", "16")
    os.environ.setdefault("MKL_NUM_THREADS", "16")
    os.environ.setdefault("OPENBLAS_NUM_THREADS", "16")

    import dai
    import numpy as np
    import pandas as pd
    import polars as pl
    import torch
    from torch import nn

    # AI应用环节：20个分钟微观结构因子和5个PIT财务因子形成日token，
    # 小型Transformer学习10个交易日内的非线性状态演化。
    minute_features = [
        "f01_body_strength",
        "f01_shadow_asymmetry",
        "f02_flow_path_confirm",
        "f03_down_jump_share",
        "f03_late_stress",
        "f05_log_price_slope",
        "f06_underwater_share",
        "f07_roughness",
        "f08_market_corr",
        "f10_amihud_intraday",
        "f10_kyle_lambda",
        "f11_cycle_1_energy",
        "f11_cycle_2_energy",
        "f12_book_imbalance",
        "f13_ret_amount_corr",
        "f13_signed_amount_pressure",
        "f17_direction_consistency",
        "f19_down_up_variance_logratio",
        "f19_leverage_corr",
        "f21_amount_time_center",
    ]
    finance_features = [
        "fin_current_ratio",
        "fin_low_accrual",
        "fin_cash_roa",
        "fin_revenue_yoy",
        "fin_low_leverage",
    ]
    features = minute_features + finance_features
    minute_columns = [
        "date",
        "instrument_id",
        "adjust_factor",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "amount",
        "ask_price1",
        "bid_price1",
        "ask_volume1",
        "ask_volume2",
        "ask_volume3",
        "bid_volume1",
        "bid_volume2",
        "bid_volume3",
    ]
    # The competition runner normally injects a mapping, but some code-tab
    # runners inject only the primary bar table (or pass its name directly).
    # All three tables are part of the competition data contract, so resolve
    # missing optional aliases to their official names instead of raising a
    # KeyError before any query starts.
    def source_table(key, default):
        if isinstance(datasources, dict):
            value = datasources.get(key)
            if value is not None and str(value).strip():
                return str(value)
        if key == "bar1m" and isinstance(datasources, str) and datasources.strip():
            return datasources.strip()
        return default

    bar_table = source_table("bar1m", "bigalpha_2026_stock_bar1m")
    financial_table = source_table("financial", "bigalpha_2026_financial")
    universe_table = source_table("instruments", "bigalpha_2026_instruments")
    train_start = pd.Timestamp("2019-01-01")
    train_end = pd.Timestamp("2024-12-31")
    requested_start = pd.Timestamp(start_date).normalize()
    requested_end = pd.Timestamp(end_date).normalize()
    if requested_start > requested_end:
        raise ValueError("start_date must not be after end_date")

    window = 10
    dim = 32
    heads = 4
    ff_dim = 64
    layers = 1
    dropout = 0.10
    lr = 1e-3
    weight_decay = 1e-4
    seed = 114514
    min_stocks = 100
    # L1_0689在离线验证中的固定最佳轮次为第3轮。
    epochs = 3

    def query(sql, left, right):
        return dai.query(
            sql,
            filters={
                "date": [
                    pd.Timestamp(left).strftime("%Y-%m-%d 00:00:00"),
                    pd.Timestamp(right).strftime("%Y-%m-%d 23:59:59"),
                ]
            },
            compression=True,
        ).df()

    selected_sql = ",\n       ".join(f"b.{column}" for column in minute_columns)
    minute_sql = f"""
    SELECT {selected_sql}
    FROM {bar_table} b
    INNER JOIN {universe_table} i
      ON CAST(b.date AS DATE) = i.date
     AND b.instrument = i.instrument
    ORDER BY b.instrument_id, b.date
    """
    mapping_sql = f"""
    SELECT CAST(b.date AS DATE) AS date, b.instrument_id, b.instrument
    FROM {bar_table} b
    INNER JOIN {universe_table} i
      ON CAST(b.date AS DATE) = i.date
     AND b.instrument = i.instrument
    WHERE CAST(b.date AS TIME) = TIME '09:31:00'
    ORDER BY b.date, b.instrument_id
    """

    def finite_div(numerator, denominator, eps=1e-18):
        return (
            pl.when(
                numerator.is_finite()
                & denominator.is_finite()
                & (denominator.abs() > eps)
            )
            .then(numerator / denominator)
            .otherwise(None)
        )

    def compute_daily(raw):
        missing = sorted(set(minute_columns) - set(raw.columns))
        if missing:
            raise RuntimeError(f"bar1m is missing required columns: {missing}")
        raw["instrument_id"] = raw["instrument_id"].astype("int16", copy=False)
        for column in [
            "adjust_factor", "open", "high", "low", "close", "amount",
            "ask_price1", "bid_price1",
        ]:
            raw[column] = raw[column].astype("float32", copy=False)
        for column in [
            "volume", "ask_volume1", "ask_volume2", "ask_volume3",
            "bid_volume1", "bid_volume2", "bid_volume3",
        ]:
            raw[column] = raw[column].astype("int32", copy=False)

        bars = pl.from_pandas(raw, rechunk=False).sort(["instrument_id", "date"])
        bars = bars.with_columns(pl.col("date").dt.date().alias("trade_date"))
        keys = ["trade_date", "instrument_id"]
        bars = bars.with_columns(
            (pl.col("close").cum_count().over(keys) - 1).cast(pl.Int16).alias("position"),
            pl.len().over(keys).cast(pl.Int16).alias("group_size"),
            (pl.col("close") / pl.col("close").shift(1).over(keys) - 1.0)
            .cast(pl.Float32)
            .alias("minute_return"),
        )
        bars = bars.with_columns(
            finite_div(
                pl.col("position").cast(pl.Float32),
                (pl.col("group_size") - 1).clip(lower_bound=1).cast(pl.Float32),
            ).cast(pl.Float32).alias("t_norm"),
            pl.col("minute_return").shift(-1).over(keys).alias("next_return"),
            pl.col("minute_return").mean().over("date").alias("market_return"),
            (pl.col("bid_volume1") + pl.col("bid_volume2") + pl.col("bid_volume3"))
            .cast(pl.Float64).alias("bid_volume3_sum"),
            (pl.col("ask_volume1") + pl.col("ask_volume2") + pl.col("ask_volume3"))
            .cast(pl.Float64).alias("ask_volume3_sum"),
        )
        bars = bars.with_columns(
            finite_div(
                pl.col("bid_volume3_sum") - pl.col("ask_volume3_sum"),
                pl.col("bid_volume3_sum") + pl.col("ask_volume3_sum"),
            ).cast(pl.Float32).alias("book_imbalance"),
            finite_div(
                pl.col("ask_price1") - pl.col("bid_price1"),
                pl.col("ask_price1") + pl.col("bid_price1"),
            ).cast(pl.Float32).alias("relative_spread"),
            finite_div(pl.col("minute_return").abs(), pl.col("amount"))
            .cast(pl.Float32).alias("minute_illiquidity"),
            finite_div(
                pl.col("close") - pl.col("close").cum_max().over(keys),
                pl.col("close").cum_max().over(keys),
            ).cast(pl.Float32).alias("drawdown"),
            pl.col("minute_return").std().over(keys).alias("return_std"),
            pl.col("amount").median().over(keys).alias("amount_median"),
        )
        bars = bars.with_columns(
            pl.col("minute_return").abs().alias("abs_return"),
            (pl.col("minute_return").sign() * pl.col("amount")).alias("signed_amount"),
            (pl.col("minute_return").sign() * pl.col("volume"))
            .cast(pl.Float32)
            .alias("signed_volume"),
            (
                pl.col("minute_return")
                * (2.0 * math.pi * pl.col("t_norm")).sin()
            ).cast(pl.Float32).alias("cycle1_sin"),
            (
                pl.col("minute_return")
                * (2.0 * math.pi * pl.col("t_norm")).cos()
            ).cast(pl.Float32).alias("cycle1_cos"),
            (
                pl.col("minute_return")
                * (4.0 * math.pi * pl.col("t_norm")).sin()
            ).cast(pl.Float32).alias("cycle2_sin"),
            (
                pl.col("minute_return")
                * (4.0 * math.pi * pl.col("t_norm")).cos()
            ).cast(pl.Float32).alias("cycle2_cos"),
            (pl.col("amount") * pl.col("t_norm")).alias("amount_time"),
            (pl.col("drawdown") < -1e-12).cast(pl.Float32).alias("underwater"),
            (pl.col("minute_return") < -2.0 * pl.col("return_std"))
            .cast(pl.Float32).alias("down_jump"),
            pl.when(
                (pl.col("minute_return") != 0)
                & (pl.col("market_return") != 0)
                & pl.col("minute_return").is_not_null()
                & pl.col("market_return").is_not_null()
            )
            .then(
                (pl.col("minute_return").sign() == pl.col("market_return").sign())
                .cast(pl.Float32)
            )
            .otherwise(None)
            .alias("market_sign_agree"),
            pl.col("close").log().alias("log_close"),
            pl.col("next_return").abs().alias("next_abs_return"),
        )

        first30 = pl.col("position") < 30
        last30 = pl.col("position") >= pl.col("group_size") - 30
        last60 = pl.col("position") >= pl.col("group_size") - 60
        high_amount = pl.col("amount") > pl.col("amount_median")
        positive = pl.col("minute_return") > 0
        negative = pl.col("minute_return") < 0
        daily = bars.group_by(keys, maintain_order=True).agg(
            pl.col("open").first().alias("open_first"),
            pl.col("close").last().alias("close_last"),
            pl.col("high").max().alias("high_max"),
            pl.col("low").min().alias("low_min"),
            pl.col("adjust_factor").first().alias("adjust_factor_first"),
            pl.col("minute_return").sum().alias("return_sum"),
            pl.col("abs_return").sum().alias("abs_return_sum"),
            pl.col("amount").sum().alias("amount_sum"),
            pl.col("signed_amount").sum().alias("signed_amount_sum"),
            pl.col("signed_volume").count().alias("kyle_n"),
            pl.col("signed_volume").sum().alias("kyle_x_sum"),
            pl.col("minute_return").filter(pl.col("signed_volume").is_not_null())
            .sum().alias("kyle_y_sum"),
            (pl.col("signed_volume") ** 2).sum().alias("kyle_xx_sum"),
            (pl.col("signed_volume") * pl.col("minute_return"))
            .sum().alias("kyle_xy_sum"),
            pl.col("cycle1_sin").sum().alias("cycle1_sin_sum"),
            pl.col("cycle1_cos").sum().alias("cycle1_cos_sum"),
            pl.col("cycle2_sin").sum().alias("cycle2_sin_sum"),
            pl.col("cycle2_cos").sum().alias("cycle2_cos_sum"),
            pl.col("amount_time").sum().alias("amount_time_sum"),
            pl.col("underwater").mean().alias("underwater_share"),
            pl.col("down_jump").mean().alias("down_jump_share"),
            pl.col("book_imbalance").mean().alias("book_mean"),
            pl.col("minute_illiquidity").mean().alias("amihud_mean"),
            pl.col("market_sign_agree").mean().alias("market_sign_agreement"),
            pl.when(first30).then(pl.col("relative_spread")).otherwise(None)
            .mean().alias("first30_spread"),
            pl.when(last30).then(pl.col("relative_spread")).otherwise(None)
            .mean().alias("last30_spread"),
            pl.when(last60).then(pl.col("minute_return")).otherwise(0.0)
            .sum().alias("last60_return_sum"),
            pl.when(high_amount).then(pl.col("minute_return")).otherwise(0.0)
            .sum().alias("ret_high_amount_sum"),
            pl.when(~high_amount).then(pl.col("minute_return")).otherwise(0.0)
            .sum().alias("ret_low_amount_sum"),
            pl.when(positive).then(pl.col("minute_return") ** 2).otherwise(0.0)
            .sum().alias("upside_sq_sum"),
            pl.when(negative).then(pl.col("minute_return") ** 2).otherwise(0.0)
            .sum().alias("downside_sq_sum"),
            positive.cast(pl.Float32).mean().alias("positive_share"),
            pl.col("t_norm").count().alias("slope_n"),
            pl.col("t_norm").sum().alias("t_sum"),
            (pl.col("t_norm") ** 2).sum().alias("t_sq_sum"),
            pl.col("log_close").sum().alias("log_close_sum"),
            (pl.col("t_norm") * pl.col("log_close")).sum().alias("t_log_close_sum"),
            pl.corr("minute_return", "market_return").alias("market_corr"),
            pl.corr("book_imbalance", "next_return").alias("book_lead_corr"),
            pl.corr("minute_return", "amount").alias("ret_amount_corr"),
            pl.corr("minute_return", "next_abs_return").alias("leverage_corr"),
            pl.corr("abs_return", "next_abs_return").alias("absret_autocorr"),
        )
        daily = daily.with_columns(
            (pl.col("high_max") - pl.col("low_min")).alias("day_range"),
            finite_div(pl.col("signed_amount_sum"), pl.col("amount_sum"))
            .alias("signed_amount_pressure"),
            finite_div(pl.col("return_sum").abs(), pl.col("abs_return_sum"))
            .alias("path_efficiency"),
            finite_div(
                pl.col("slope_n") * pl.col("t_log_close_sum")
                - pl.col("t_sum") * pl.col("log_close_sum"),
                pl.col("slope_n") * pl.col("t_sq_sum") - pl.col("t_sum") ** 2,
            ).alias("log_price_slope"),
            finite_div(
                pl.col("kyle_n") * pl.col("kyle_xy_sum")
                - pl.col("kyle_x_sum") * pl.col("kyle_y_sum"),
                pl.col("kyle_n") * pl.col("kyle_xx_sum")
                - pl.col("kyle_x_sum") ** 2,
            ).alias("kyle_lambda"),
            finite_div(
                (
                    pl.col("cycle1_sin_sum") ** 2
                    + pl.col("cycle1_cos_sum") ** 2
                ).sqrt(),
                pl.col("abs_return_sum"),
            ).alias("cycle1_energy"),
            finite_div(
                (
                    pl.col("cycle2_sin_sum") ** 2
                    + pl.col("cycle2_cos_sum") ** 2
                ).sqrt(),
                pl.col("abs_return_sum"),
            ).alias("cycle2_energy"),
        )
        daily = daily.with_columns(
            finite_div(
                pl.col("close_last") - pl.col("open_first"), pl.col("day_range")
            ).alias("f01_body_strength"),
            finite_div(
                (pl.col("high_max") - pl.max_horizontal("open_first", "close_last"))
                - (pl.min_horizontal("open_first", "close_last") - pl.col("low_min")),
                pl.col("day_range"),
            ).alias("f01_shadow_asymmetry"),
            (pl.col("signed_amount_pressure") * pl.col("path_efficiency"))
            .alias("f02_flow_path_confirm"),
            pl.col("down_jump_share").alias("f03_down_jump_share"),
            (
                (-pl.col("last60_return_sum")).clip(lower_bound=0.0)
                * (
                    finite_div(pl.col("last30_spread"), pl.col("first30_spread"))
                    - 1.0
                ).clip(lower_bound=0.0)
            ).alias("f03_late_stress"),
            pl.col("log_price_slope").alias("f05_log_price_slope"),
            pl.col("underwater_share").alias("f06_underwater_share"),
            finite_div(
                pl.col("abs_return_sum"), pl.col("return_sum").abs() + 1e-8
            ).log1p().alias("f07_roughness"),
            pl.col("market_corr").alias("f08_market_corr"),
            pl.col("amihud_mean").alias("f10_amihud_intraday"),
            pl.col("kyle_lambda").alias("f10_kyle_lambda"),
            pl.col("cycle1_energy").alias("f11_cycle_1_energy"),
            pl.col("cycle2_energy").alias("f11_cycle_2_energy"),
            pl.col("book_mean").alias("f12_book_imbalance"),
            pl.col("ret_amount_corr").alias("f13_ret_amount_corr"),
            pl.col("signed_amount_pressure").alias("f13_signed_amount_pressure"),
            pl.when(pl.col("return_sum") >= 0)
            .then(pl.col("positive_share"))
            .otherwise(1.0 - pl.col("positive_share"))
            .alias("f17_direction_consistency"),
            (
                (
                    pl.col("downside_sq_sum")
                    + (pl.col("downside_sq_sum") + pl.col("upside_sq_sum")) * 1e-8
                    + 1e-20
                )
                / (
                    pl.col("upside_sq_sum")
                    + (pl.col("downside_sq_sum") + pl.col("upside_sq_sum")) * 1e-8
                    + 1e-20
                )
            ).log().alias("f19_down_up_variance_logratio"),
            pl.col("leverage_corr").alias("f19_leverage_corr"),
            finite_div(pl.col("amount_time_sum"), pl.col("amount_sum"))
            .alias("f21_amount_time_center"),
            (pl.col("open_first") * pl.col("adjust_factor_first"))
            .alias("adjusted_open"),
        )
        return daily.select(
            pl.col("trade_date").cast(pl.Date).alias("date"),
            pl.col("instrument_id").cast(pl.Int16),
            pl.col("adjusted_open").cast(pl.Float32),
            *[pl.col(column).cast(pl.Float32) for column in minute_features],
        ).sort(["date", "instrument_id"])

    # 只查询训练区间和私榜推理所需的短pre-roll，不读取中间空白年份的分钟数据。
    ranges = [
        (pd.Timestamp(year, 1, 1), pd.Timestamp(year, 12, 31))
        for year in range(2019, 2025)
    ]
    inference_left = requested_start - pd.Timedelta(days=45)
    if requested_end > train_end:
        ranges.append((inference_left, requested_end))

    mapping_left = min(train_start, inference_left)
    data_right = max(train_end, requested_end)
    mapping = query(mapping_sql, mapping_left, data_right)
    mapping["instrument_id"] = mapping["instrument_id"].astype("int16")
    mapping["instrument"] = mapping["instrument"].astype("string")
    id_map = mapping[["instrument_id", "instrument"]].drop_duplicates()
    if id_map["instrument_id"].duplicated().any():
        raise RuntimeError("instrument_id mapping is not stable")

    daily_parts = []
    empty_ranges = []
    for left, right in ranges:
        raw = query(minute_sql, left, right)
        if raw.empty:
            # Holidays and unavailable date partitions contribute no rows.
            # The returned frame already contains only observable trading days.
            empty_ranges.append((left.date(), right.date()))
            continue
        daily_parts.append(compute_daily(raw))
        del raw
        gc.collect()
    if not daily_parts:
        raise RuntimeError("No bar1m rows in any requested range")
    minute = pl.concat(daily_parts, how="vertical").unique(
        subset=["date", "instrument_id"], keep="last"
    ).sort(["date", "instrument_id"]).to_pandas()
    minute["date"] = pd.to_datetime(minute["date"]).astype("datetime64[ns]")
    missing_ids = sorted(
        set(minute["instrument_id"].astype(int).unique())
        - set(id_map["instrument_id"].astype(int).unique())
    )
    if missing_ids:
        id_list = ",".join(str(value) for value in missing_ids)
        fallback_sql = f"""
        SELECT instrument_id, instrument
        FROM {bar_table}
        WHERE instrument_id IN ({id_list})
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY instrument_id ORDER BY date
        ) = 1
        ORDER BY instrument_id
        """
        fallback = query(fallback_sql, mapping_left, data_right)
        fallback["instrument_id"] = fallback["instrument_id"].astype("int16")
        fallback["instrument"] = fallback["instrument"].astype("string")
        id_map = pd.concat(
            [id_map, fallback[["instrument_id", "instrument"]]],
            ignore_index=True,
        ).drop_duplicates("instrument_id", keep="last")
    minute = minute.merge(id_map, on="instrument_id", how="left", validate="many_to_one")
    if minute["instrument"].isna().any():
        raise RuntimeError("Unmapped instruments in daily minute factors")
    del daily_parts, mapping
    gc.collect()

    # 财务数据严格按平台提供的PIT可用日期向后匹配，绝不按report_date前视。
    financial_sql = f"""
    SELECT date, instrument, report_date, shift, category,
           total_current_assets, total_current_liabilities, total_assets,
           interest_bearing_debt, net_cffoa,
           net_profit_to_parent_shareholders, operating_revenue
    FROM {financial_table}
    WHERE shift = 0 AND category IN ('lf', 'ttm')
    ORDER BY instrument, date, report_date
    """
    events = query(financial_sql, pd.Timestamp("2017-01-01"), data_right)
    events["date"] = pd.to_datetime(events["date"]).dt.normalize().astype("datetime64[ns]")
    events["report_date"] = pd.to_datetime(events["report_date"]).astype("datetime64[ns]")
    events["instrument"] = events["instrument"].astype("string")
    if not events["shift"].eq(0).all():
        raise RuntimeError("financial source returned non-zero shift")

    # 为私榜财务同比保留约400日连续日历，分钟数据仍只取45日pre-roll。
    calendar_ranges = [(train_start, train_end)]
    if requested_end > train_end:
        calendar_ranges.append((requested_start - pd.Timedelta(days=400), requested_end))
    calendar_parts = [
        query(
            f"SELECT date, instrument FROM {universe_table} ORDER BY date, instrument",
            left,
            right,
        )
        for left, right in calendar_ranges
    ]
    calendar = pd.concat(calendar_parts, ignore_index=True).drop_duplicates(
        ["date", "instrument"]
    )
    calendar["date"] = pd.to_datetime(calendar["date"]).dt.normalize().astype("datetime64[ns]")
    calendar["instrument"] = calendar["instrument"].astype("string")
    panel = calendar.sort_values(["date", "instrument"])
    category_fields = {
        "lf": [
            "total_current_assets", "total_current_liabilities", "total_assets",
            "interest_bearing_debt",
        ],
        "ttm": [
            "net_cffoa", "net_profit_to_parent_shareholders", "operating_revenue",
        ],
    }
    for category, fields0 in category_fields.items():
        right = events.loc[
            events["category"].eq(category),
            ["date", "instrument", "report_date", *fields0],
        ].sort_values(["date", "instrument", "report_date"])
        right = right.drop_duplicates(["date", "instrument"], keep="last")
        right = right.rename(columns={field: f"{category}_{field}" for field in fields0})
        panel = pd.merge_asof(
            panel.sort_values(["date", "instrument"]),
            right.sort_values(["date", "instrument"]),
            on="date",
            by="instrument",
            direction="backward",
            allow_exact_matches=True,
        )
    panel = panel.sort_values(["instrument", "date"]).reset_index(drop=True)
    prior_assets = panel.groupby("instrument", sort=False)["lf_total_assets"].shift(252)
    prior_revenue = panel.groupby("instrument", sort=False)["ttm_operating_revenue"].shift(252)
    average_assets = (panel["lf_total_assets"] + prior_assets) / 2.0

    def safe_div_pd(numerator, denominator):
        valid = (
            numerator.notna() & denominator.notna()
            & np.isfinite(numerator) & np.isfinite(denominator)
            & (denominator.abs() > 1e-12)
        )
        result = pd.Series(np.nan, index=numerator.index, dtype="float32")
        result.loc[valid] = (numerator.loc[valid] / denominator.loc[valid]).astype("float32")
        return result

    finance = panel[["date", "instrument"]].copy()
    finance["fin_current_ratio"] = safe_div_pd(
        panel["lf_total_current_assets"], panel["lf_total_current_liabilities"]
    )
    finance["fin_low_accrual"] = -safe_div_pd(
        panel["ttm_net_profit_to_parent_shareholders"] - panel["ttm_net_cffoa"],
        average_assets,
    )
    finance["fin_cash_roa"] = safe_div_pd(panel["ttm_net_cffoa"], average_assets)
    valid_growth = (
        panel["ttm_operating_revenue"].gt(0) & prior_revenue.gt(0)
        & panel["ttm_operating_revenue"].notna() & prior_revenue.notna()
    )
    finance["fin_revenue_yoy"] = np.nan
    finance.loc[valid_growth, "fin_revenue_yoy"] = np.log(
        panel.loc[valid_growth, "ttm_operating_revenue"] / prior_revenue.loc[valid_growth]
    )
    finance["fin_low_leverage"] = -safe_div_pd(
        panel["lf_interest_bearing_debt"], panel["lf_total_assets"]
    )
    for column in finance_features:
        finance[column] = pd.to_numeric(finance[column], errors="coerce").astype("float32")
    finance = finance[["date", "instrument", *finance_features]]
    minute = minute.merge(
        finance,
        on=["date", "instrument"],
        how="left",
        validate="one_to_one",
    )
    del events, calendar_parts, calendar, panel, finance
    gc.collect()

    dates = np.array(sorted(minute["date"].unique()), dtype="datetime64[ns]")
    instruments = np.array(sorted(minute["instrument"].astype(str).unique()), dtype=object)
    date_index = pd.Index(dates)
    instrument_index = pd.Index(instruments)
    row_days = date_index.get_indexer(minute["date"].to_numpy(dtype="datetime64[ns]"))
    row_instruments = instrument_index.get_indexer(minute["instrument"].astype(str).to_numpy())
    shape = (len(dates), len(instruments))
    raw_cube = np.full((*shape, len(features)), np.nan, dtype=np.float32)
    raw_cube[row_days, row_instruments] = minute[features].to_numpy(dtype=np.float32)
    opening = np.full(shape, np.nan, dtype=np.float32)
    opening[row_days, row_instruments] = minute["adjusted_open"].to_numpy(dtype=np.float32)
    observed = np.isfinite(raw_cube[:, :, : len(minute_features)]).any(axis=2)
    labels = np.full(shape, np.nan, dtype=np.float32)
    with np.errstate(divide="ignore", invalid="ignore"):
        labels[:-2] = opening[2:] / opening[1:-1] - 1.0
    labels[~np.isfinite(labels)] = np.nan
    train_mask = dates <= np.datetime64(train_end)
    train_values = raw_cube[train_mask].reshape(-1, len(features))
    mean = np.nanmean(train_values, axis=0).astype(np.float32)
    std = np.nanstd(train_values, axis=0, ddof=1).astype(np.float32)
    std = np.where(np.isfinite(std) & (std > 1e-6), std, 1.0).astype(np.float32)
    standardized = np.nan_to_num(
        (raw_cube - mean) / std, nan=0.0, posinf=0.0, neginf=0.0
    ).astype(np.float32)
    targets = np.full(labels.shape, np.nan, dtype=np.float32)
    for day in range(len(dates)):
        valid = np.isfinite(labels[day])
        if valid.sum() < 20:
            continue
        values = labels[day, valid].astype(np.float64)
        lo, hi = np.quantile(values, [0.01, 0.99])
        values = np.clip(values, lo, hi)
        scale = values.std(ddof=1)
        if scale > 1e-12:
            targets[day, valid] = ((values - values.mean()) / scale).astype(np.float32)

    class TinyDailyTransformer(nn.Module):
        def __init__(self):
            super().__init__()
            self.projection = nn.Linear(len(features), dim)
            self.position = nn.Parameter(torch.zeros(window, dim))
            layer = nn.TransformerEncoderLayer(
                d_model=dim,
                nhead=heads,
                dim_feedforward=ff_dim,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
            self.encoder = nn.TransformerEncoder(
                layer, num_layers=layers, enable_nested_tensor=False
            )
            self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, 1))
            nn.init.normal_(self.position, mean=0.0, std=0.02)

        def forward(self, x, padding):
            hidden = self.projection(x) + self.position.unsqueeze(0)
            hidden = self.encoder(hidden, src_key_padding_mask=padding)
            return self.head(hidden[:, -1]).squeeze(-1)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    try:
        torch.set_num_threads(min(16, os.cpu_count() or 1))
        torch.set_num_interop_threads(1)
    except RuntimeError:
        # Some evaluators initialize PyTorch before calling main.
        pass
    torch.use_deterministic_algorithms(True)
    cube_t = torch.from_numpy(standardized)
    observed_t = torch.from_numpy(observed)
    targets_t = torch.from_numpy(targets)
    train_days = []
    for day, date in enumerate(dates):
        if day < window - 1 or day + 2 >= len(dates):
            continue
        if (
            train_start <= pd.Timestamp(date) <= train_end
            and pd.Timestamp(dates[day + 2]) <= train_end
            and np.isfinite(labels[day]).sum() >= min_stocks
        ):
            train_days.append(day)
    if not train_days:
        raise RuntimeError("No eligible training days")
    model = TinyDailyTransformer()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=weight_decay, foreach=True
    )
    for epoch in range(1, epochs + 1):
        model.train()
        order = np.random.default_rng(seed + epoch).permutation(train_days)
        for day in order:
            target = targets_t[day]
            stock_ids = torch.where(torch.isfinite(target) & observed_t[day])[0]
            if stock_ids.numel() < min_stocks:
                continue
            x = cube_t[day - window + 1 : day + 1, stock_ids].permute(1, 0, 2)
            padding = ~observed_t[
                day - window + 1 : day + 1, stock_ids
            ].permute(1, 0)
            prediction = model(x, padding)
            loss = nn.functional.mse_loss(prediction, target[stock_ids])
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

    model.eval()
    prediction_frames = []
    with torch.no_grad():
        for day, date in enumerate(dates):
            timestamp = pd.Timestamp(date).normalize()
            if not (requested_start <= timestamp <= requested_end):
                continue
            if day < window - 1:
                raise RuntimeError(f"Insufficient pre-roll for {timestamp.date()}")
            stock_ids = torch.where(observed_t[day])[0]
            x = cube_t[day - window + 1 : day + 1, stock_ids].permute(1, 0, 2)
            padding = ~observed_t[
                day - window + 1 : day + 1, stock_ids
            ].permute(1, 0)
            scores = model(x, padding).cpu().numpy().astype(np.float64)
            prediction_frames.append(
                pd.DataFrame(
                    {
                        "date": timestamp,
                        "instrument": instruments[stock_ids.numpy()],
                        "factor": scores,
                    }
                )
            )
    if not prediction_frames:
        raise RuntimeError("No predictions in requested interval")
    predictions = pd.concat(prediction_frames, ignore_index=True)

    # 返回完整官方股票池。少量停牌/缺分钟数据股票填0（截面中性值），避免缺日或覆盖不足。
    output_pool = query(
        f"SELECT date, instrument FROM {universe_table} ORDER BY date, instrument",
        requested_start,
        requested_end,
    )
    output_pool["date"] = pd.to_datetime(output_pool["date"]).dt.normalize()
    output_pool["instrument"] = output_pool["instrument"].astype("string")
    result = output_pool.merge(
        predictions,
        on=["date", "instrument"],
        how="left",
        validate="one_to_one",
    )
    coverage = result.groupby("date")["factor"].apply(lambda values: values.notna().mean())
    if coverage.empty or coverage.min() < 0.60:
        raise RuntimeError(f"Prediction coverage below 60%: {coverage.to_dict()}")
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce")
    result.loc[~np.isfinite(result["factor"]), "factor"] = np.nan
    result["factor"] = result.groupby("date")["factor"].transform(
        lambda values: values.fillna(values.median())
    ).fillna(0.0)
    result = result[["date", "instrument", "factor"]].sort_values(
        ["date", "instrument"]
    ).reset_index(drop=True)
    if list(result.columns) != ["date", "instrument", "factor"]:
        raise RuntimeError("Invalid output schema")
    if result.duplicated(["date", "instrument"]).any():
        raise RuntimeError("Duplicate output keys")
    if not np.isfinite(result["factor"]).all():
        raise RuntimeError("Non-finite output factor")
    return result

